# 03 - Inference Demo (Two-Stage)
Demo end-to-end cu detector + classifier pe un folder de imagini.

In [1]:
import sys
import subprocess
from pathlib import Path

REPO_ROOT = Path('../..').resolve()
py = sys.executable

DETECTOR_PT = REPO_ROOT / 'runs' / 'detect' / 'parks-trash-A22' / 'weights' / 'best.pt'
CLASSIFIER_PT = REPO_ROOT / 'runs' / 'classify' / 'parks-cls-B2' / 'weights' / 'best.pt'
SOURCE_DIR = REPO_ROOT / 'datasets' / 'parks_detect' / 'images' / 'test'
OUT_DIR = REPO_ROOT / 'outputs' / 'two_stage_demo'
DEVICE = '0'

print('detector  :', DETECTOR_PT)
print('classifier:', CLASSIFIER_PT)
print('source    :', SOURCE_DIR)

detector  : D:\TrashDetectionSystem\runs\detect\parks-trash-A22\weights\best.pt
classifier: D:\TrashDetectionSystem\runs\classify\parks-cls-B2\weights\best.pt
source    : D:\TrashDetectionSystem\datasets\parks_detect\images\test


In [2]:
assert SOURCE_DIR.exists(), f'Missing source dir: {SOURCE_DIR}'
assert DETECTOR_PT.exists(), f'Missing detector: {DETECTOR_PT}'
assert CLASSIFIER_PT.exists(), f'Missing classifier: {CLASSIFIER_PT}'

In [ ]:
import subprocess as _sp

cmd = [
    py, 'scripts/run_two_stage_batch.py',
    '--source-dir', str(SOURCE_DIR),
    '--detector', str(DETECTOR_PT),
    '--classifier', str(CLASSIFIER_PT),
    '--det-imgsz', '416',
    '--cls-imgsz', '224',
    '--device', DEVICE,
    '--save-images',
    '--out-dir', str(OUT_DIR)
]
print('> ' + ' '.join(cmd))

proc = _sp.Popen(cmd, cwd=str(REPO_ROOT),
                 stdout=_sp.PIPE, stderr=_sp.STDOUT,
                 text=True, bufsize=1, encoding='utf-8', errors='replace')
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f'demo failed: {proc.returncode}')


> d:\TrashDetectionSystem\.venv\Scripts\python.exe scripts/run_two_stage_batch.py --source-dir D:\TrashDetectionSystem\datasets\parks_detect\images\test --detector D:\TrashDetectionSystem\runs\detect\parks-trash-A22\weights\best.pt --classifier D:\TrashDetectionSystem\runs\classify\parks-cls-B2\weights\best.pt --det-imgsz 416 --cls-imgsz 224 --device 0 --save-images --out-dir D:\TrashDetectionSystem\outputs\two_stage_demo


RuntimeError: demo failed: 1

In [ ]:
from IPython.display import Image, display

annotated = sorted((OUT_DIR / 'annotated').glob('*.*'))
print('annotated images:', len(annotated))
for p in annotated[:5]:
    display(Image(filename=str(p), width=700))

# 03 — Inferență Two-Stage & Demo Vizual

Rulează **pipeline-ul complet two-stage** pe imagini individuale sau pe un director întreg:

1. **Stage 1** — Detector YOLO localizează obiectele `trash`
2. **Stage 2** — Clasificator YOLO prezice materialul fiecărui obiect

Rezultate:
- Imagini adnotate cu bounding box + material
- CSV cu toate detecțiile
- Distribuție materiale vizualizată

**Pre-condiție**: Ai antrenat detectorul (Exp A2/A3) și clasificatorul (Exp B3).

In [4]:
import csv
import sys
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from ultralytics import YOLO
from src.detect_two_stage import detect_and_classify, draw_detections, classifier_names

print(f"Rădăcina proiectului: {REPO_ROOT}")

OSError: [WinError 1455] The paging file is too small for this operation to complete. Error loading "d:\TrashDetectionSystem\.venv\Lib\site-packages\torch\lib\shm.dll" or one of its dependencies.

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────

DETECTOR_PT   = REPO_ROOT / "runs" / "detect"   / "parks-trash-A22" / "weights" / "best.pt"
CLASSIFIER_PT = REPO_ROOT / "runs" / "classify" / "parks-cls-B2"   / "weights" / "best.pt"

# Sursă imagini pentru inferență demo
SOURCE_DIR    = REPO_ROOT / "datasets" / "parks_detect" / "images" / "test"

# Parametri inferență
DET_CONF      = 0.25
DET_IMGSZ     = 416         # trebuie să coincidă cu imgsz din antrenare A2
CLS_IMGSZ     = 224
MAX_LABELS    = 5
LINE_WIDTH    = 2
DEVICE        = "0"         # "0" = GPU RTX 3050 | "cpu"

# Output
OUTPUT_DIR    = REPO_ROOT / "outputs" / "two_stage_demo"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_EXTS    = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print(f"Detector  : {DETECTOR_PT}")
print(f"Clasificor: {CLASSIFIER_PT}")
print(f"Sursă     : {SOURCE_DIR}")


---
## Încărcare modele

In [ ]:
assert DETECTOR_PT.exists(),   f"Detector lipsă: {DETECTOR_PT}"
assert CLASSIFIER_PT.exists(), f"Clasificator lipsă: {CLASSIFIER_PT}"

detector   = YOLO(str(DETECTOR_PT))
classifier = YOLO(str(CLASSIFIER_PT))
cls_names  = classifier_names(classifier)

print(f"Clase clasificator: {cls_names}")

---
## Inferență pe un director de imagini

In [ ]:
images = sorted(p for p in SOURCE_DIR.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS)
print(f"Imagini găsite: {len(images)}")

all_detections = []
material_counter = Counter()

csv_path = OUTPUT_DIR / "detections.csv"
with csv_path.open("w", encoding="utf-8", newline="") as fh:
    writer = csv.writer(fh)
    writer.writerow(["image_path","detection_index","x1","y1","x2","y2",
                     "det_score","material_name","material_score"])

    for img_path in images:
        frame = cv2.imread(str(img_path))
        if frame is None: continue

        detections = detect_and_classify(
            frame, detector, classifier,
            DET_CONF, DET_IMGSZ, CLS_IMGSZ, cls_names
        )
        all_detections.append((img_path, frame, detections))
        for d in detections:
            material_counter[d["material_name"]] += 1
            l, t, r, b = d["box"]
            writer.writerow([img_path.as_posix(), d["index"], l, t, r, b,
                             d["det_score"], d["material_name"], d["material_score"]])

total_det = sum(len(d[2]) for d in all_detections)
print(f"Imagini procesate  : {len(all_detections)}")
print(f"Total detecții     : {total_det}")
print(f"CSV salvat         : {csv_path}")
print(f"\nDistribuție materiale:")
for mat, cnt in material_counter.most_common():
    print(f"  {mat:<10}: {cnt}")

---
## Distribuție Materiale Detectate

In [ ]:
if material_counter:
    labels = list(material_counter.keys())
    values = list(material_counter.values())
    colors = ["#4e9af1","#f17c4e","#62c370","#f1c94e","#a16af1"]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # Bar chart
    ax1.bar(labels, values, color=colors[:len(labels)], edgecolor="white", linewidth=0.7)
    ax1.set_title("Număr detecții per material")
    ax1.set_ylabel("Detecții")
    ax1.set_xlabel("Material")
    ax1.tick_params(axis="x", rotation=15)

    # Pie chart
    ax2.pie(values, labels=labels, colors=colors[:len(labels)], autopct="%1.1f%%",
            startangle=140, pctdistance=0.8)
    ax2.set_title("Distribuție procentuală materiale")

    plt.suptitle(f"Two-Stage — {len(images)} imagini  |  {total_det} detecții totale", fontsize=12)
    plt.tight_layout()

    chart_path = OUTPUT_DIR / "material_distribution.png"
    plt.savefig(str(chart_path), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Grafic salvat: {chart_path}")
else:
    print("Nicio detecție — verifică pragul de confidență (DET_CONF) sau modelul.")

---
## Vizualizare imagini adnotate (Two-Stage)
Afișează bounding box-uri + material + confidence pentru primele N imagini.

In [ ]:
N_PREVIEW = 6
MATERIAL_COLORS = {
    "glass":   (78,  154, 241),
    "metal":   (241, 124,  78),
    "other":   ( 98, 195, 112),
    "paper":   (241, 201,  78),
    "plastic": (161, 106, 241),
    "unknown": (200, 200, 200),
}

preview_samples = [s for s in all_detections if s[2]][:N_PREVIEW]
if not preview_samples:
    preview_samples = all_detections[:N_PREVIEW]

if not preview_samples:
    print("Nicio imagine disponibilă pentru preview.")
else:
    cols = 3
    n = len(preview_samples)
    rows_n = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows_n, cols, figsize=(16, 5*rows_n))
    axes = np.array(axes).flatten()

    for ax, (img_path, frame, dets) in zip(axes, preview_samples):
        annotated = draw_detections(frame.copy(), dets, fps=0.0, max_labels=MAX_LABELS, line_width=LINE_WIDTH)
        ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        ax.set_title(f"{img_path.name}  [{len(dets)} det.]", fontsize=9)
        ax.axis("off")

    for ax in axes[n:]:
        ax.axis("off")

    plt.suptitle("Rezultate Two-Stage — Detector + Clasificator Materiale", fontsize=13)
    plt.tight_layout()
    plt.show()

---
## Salvare imagini adnotate

In [ ]:
annotated_dir = OUTPUT_DIR / "annotated"
annotated_dir.mkdir(exist_ok=True)

saved = 0
for img_path, frame, dets in all_detections:
    annotated = draw_detections(frame.copy(), dets, fps=0.0, max_labels=MAX_LABELS, line_width=LINE_WIDTH)
    cv2.imwrite(str(annotated_dir / img_path.name), annotated)
    saved += 1

print(f"{saved} imagini adnotate salvate în: {annotated_dir}")

---
## Inferență pe o singură imagine
Testează rapid pe orice imagine de pe disc.

In [ ]:
# Schimbă calea cu o imagine de test
SINGLE_IMAGE = REPO_ROOT / "datasets" / "parks_detect" / "images" / "test"

# Caută prima imagine disponibilă
candidates = sorted(SINGLE_IMAGE.iterdir()) if SINGLE_IMAGE.is_dir() else []
first_img = next((p for p in candidates if p.suffix.lower() in IMAGE_EXTS), None)

if first_img is None:
    print("Nicio imagine de test găsită. Schimbă SINGLE_IMAGE cu o cale validă.")
else:
    frame = cv2.imread(str(first_img))
    dets  = detect_and_classify(frame, detector, classifier, DET_CONF, DET_IMGSZ, CLS_IMGSZ, cls_names)
    ann   = draw_detections(frame.copy(), dets, fps=0.0, max_labels=5, line_width=2)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Original")
    axes[0].axis("off")
    axes[1].imshow(cv2.cvtColor(ann, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f"Two-Stage ({len(dets)} detecții)")
    axes[1].axis("off")
    plt.suptitle(first_img.name, fontsize=11)
    plt.tight_layout()
    plt.show()

    print(f"\nDetecții ({len(dets)}):")
    for d in dets:
        print(f"  [{d['index']}] {d['material_name']:<10} conf={d['material_score']:.2f}  box={d['box']}")